### 1. Importar librerías y paquetes

In [1]:
# Importar librerias
import pandas as pd
import unicodedata
import os

### 2. Importar datos de origen

#### 2.1. Importar datos de Provincias

In [2]:
# Importar datos de Provincias (del INE)
df_provincias = pd.read_excel(
    "../Data/Raw/Public/INE/D_provincias_ine_prev.xlsx"
)

# Visualizar los resultados
df_provincias.head(5)

,id_provincia,cod-ca,cod-prv,nombre_provincia,nombre_provincia_completo
0,01-04,1,4,Almería,01-04 - Almería
1,01-11,1,11,Cádiz,01-11 - Cádiz
2,01-14,1,14,Córdoba,01-14 - Córdoba
3,01-18,1,18,Granada,01-18 - Granada
4,01-21,1,21,Huelva,01-21 - Huelva


#### 2.2. Importar datos de CCAA

In [3]:
# Importar datos de CCAA (del INE)
df_ccaa = pd.read_excel(
    "../Data/Raw/Public/INE/D_ccaa_ine_prev.xlsx"
)

# Visualizar los resultados
df_ccaa.head(5)

,id,nombre,nombre_completo
0,1,Andalucía,01 Andalucía
1,2,Aragón,02 Aragón
2,3,Asturias,03 Asturias
3,4,Balears,04 Balears
4,5,Canarias,05 Canarias


### 3. Transformación de datos (tablas dimensiones)

#### 3.1 Tabla Dimensiones Provincias 

In [4]:
# Sacar tildes del campo nombre_provincia
df_provincias["nombre_provincia_st"] = df_provincias["nombre_provincia"].apply(
    lambda x: ''.join(
        c for c in unicodedata.normalize('NFD', x)
        if unicodedata.category(c) != 'Mn'
    )
)

# Renombrar columnas
df_provincias = df_provincias.rename(columns={
    "id_provincia": "Id_provincia",
    "cod-ca": "Id_ccaa",
    "cod-prv": "Cod-prv",
    "nombre_provincia": "Nombre_provincia",
    "nombre_provincia_st": "Nombre_provincia_st",
    "nombre_provincia_completo": "Nombre_provincia_completo"    
})

# Ordenar las columnas
df_provincias = df_provincias[['Id_provincia', 'Id_ccaa', 'Cod-prv', 'Nombre_provincia', 'Nombre_provincia_st',
       'Nombre_provincia_completo']]

# Visualizar los resultados
df_provincias

,Id_provincia,Id_ccaa,Cod-prv,Nombre_provincia,Nombre_provincia_st,Nombre_provincia_completo
0,01-04,1,4,Almería,Almeria,01-04 - Almería
1,01-11,1,11,Cádiz,Cadiz,01-11 - Cádiz
2,01-14,1,14,Córdoba,Cordoba,01-14 - Córdoba
3,01-18,1,18,Granada,Granada,01-18 - Granada
4,01-21,1,21,Huelva,Huelva,01-21 - Huelva
5,01-23,1,23,Jaén,Jaen,01-23 - Jaén
6,01-29,1,29,Málaga,Malaga,01-29 - Málaga
7,01-41,1,41,Sevilla,Sevilla,01-41 - Sevilla
8,02-22,2,22,Huesca,Huesca,02-22 - Huesca
9,02-44,2,44,Teruel,Teruel,02-44 - Teruel


#### 3.2 Tabla Dimensiones CCAA

In [5]:
# Renombrar columnas
df_ccaa = df_ccaa.rename(columns={
    "id": "Id_ccaa",
    "nombre": "Nombre_ccaa",
    "nombre_completo": "Nombre_ccaa_completo" 
})

# Visualizar los resultados
df_ccaa.head(5)

,Id_ccaa,Nombre_ccaa,Nombre_ccaa_completo
0,1,Andalucía,01 Andalucía
1,2,Aragón,02 Aragón
2,3,Asturias,03 Asturias
3,4,Balears,04 Balears
4,5,Canarias,05 Canarias


#### 3.3 Tabla Dimensiones Geografía

In [6]:
#Join DFs (provincia y ccaa)
df_geografia = df_provincias.merge(
    df_ccaa[
        ["Id_ccaa","Nombre_ccaa","Nombre_ccaa_completo"]
    ],
    how="left",
    left_on="Id_ccaa",
    right_on="Id_ccaa"
)

#Reordenar columnas del DF
df_geografia = df_geografia[['Id_provincia', 'Cod-prv', 'Nombre_provincia',
       'Nombre_provincia_st', 'Nombre_provincia_completo', 'Id_ccaa','Nombre_ccaa','Nombre_ccaa_completo']]

#Visualizar los resultados
df_geografia

,Id_provincia,Cod-prv,Nombre_provincia,Nombre_provincia_st,Nombre_provincia_completo,Id_ccaa,Nombre_ccaa,Nombre_ccaa_completo
0,01-04,4,Almería,Almeria,01-04 - Almería,1,Andalucía,01 Andalucía
1,01-11,11,Cádiz,Cadiz,01-11 - Cádiz,1,Andalucía,01 Andalucía
2,01-14,14,Córdoba,Cordoba,01-14 - Córdoba,1,Andalucía,01 Andalucía
3,01-18,18,Granada,Granada,01-18 - Granada,1,Andalucía,01 Andalucía
4,01-21,21,Huelva,Huelva,01-21 - Huelva,1,Andalucía,01 Andalucía
5,01-23,23,Jaén,Jaen,01-23 - Jaén,1,Andalucía,01 Andalucía
6,01-29,29,Málaga,Malaga,01-29 - Málaga,1,Andalucía,01 Andalucía
7,01-41,41,Sevilla,Sevilla,01-41 - Sevilla,1,Andalucía,01 Andalucía
8,02-22,22,Huesca,Huesca,02-22 - Huesca,2,Aragón,02 Aragón
9,02-44,44,Teruel,Teruel,02-44 - Teruel,2,Aragón,02 Aragón


### 4. Exportar los datos a Excel

#### 4.1 Almacenamiento Tabla Dimensiones Provincias

In [7]:
#Exportar los resultados a Excel
df_provincias.to_excel("../Data/Silver/D_provincias_ine.xlsx", index=False)

#### 4.2 Almacenamiento Tabla Dimensiones CCAA

In [8]:
#Exportar los resultados a Excel
df_ccaa.to_excel("../Data/Silver/D_ccaa_ine.xlsx", index=False)

#### 4.3 Almacenamiento Tabla Dimensiones Geografía

In [9]:
#Exportar los resultados a Excel
df_geografia.to_excel("../Data/Gold/Dim/D_Geografia.xlsx", index=False)